# Environment & Data Inventor 

In [1]:
# ============================================================
# Environment & Data Inventory (ONE CELL)
# - Minimal, safe checks for Kaggle input structure
# - Loads core CSVs (train/valid/test + labels)
# - Verifies folders (MSA, PDB_RNA, extra) and key extra files
# - Prints compact summary (no heavy parsing)
# ============================================================

import os
from pathlib import Path
import pandas as pd

# ---------- Paths ----------
COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
assert COMP_ROOT.exists(), f"COMP_ROOT not found: {COMP_ROOT}"

PATHS = {
    "train_sequences": COMP_ROOT / "train_sequences.csv",
    "validation_sequences": COMP_ROOT / "validation_sequences.csv",
    "test_sequences": COMP_ROOT / "test_sequences.csv",
    "train_labels": COMP_ROOT / "train_labels.csv",
    "validation_labels": COMP_ROOT / "validation_labels.csv",
    "sample_submission": COMP_ROOT / "sample_submission.csv",
    "MSA_dir": COMP_ROOT / "MSA",
    "PDB_RNA_dir": COMP_ROOT / "PDB_RNA",
    "extra_dir": COMP_ROOT / "extra",
    # common extra files (do not assume all exist)
    "extra_parse_fasta": COMP_ROOT / "extra" / "parse_fasta_py.py",
    "extra_rna_metadata": COMP_ROOT / "extra" / "rna_metadata.csv",
    "extra_readme": COMP_ROOT / "extra" / "README.md",
    # common PDB helper files
    "pdb_seqres_fasta": COMP_ROOT / "PDB_RNA" / "pdb_seqres_NA.fasta",
    "pdb_release_dates": COMP_ROOT / "PDB_RNA" / "pdb_release_dates_NA.csv",
}

# ---------- Load core CSVs ----------
train_seqs = pd.read_csv(PATHS["train_sequences"])
valid_seqs = pd.read_csv(PATHS["validation_sequences"])
test_seqs  = pd.read_csv(PATHS["test_sequences"])
train_labels = pd.read_csv(PATHS["train_labels"])
valid_labels = pd.read_csv(PATHS["validation_labels"])
sample_sub = pd.read_csv(PATHS["sample_submission"])

# ---------- Quick sanity checks ----------
def _require_cols(df, cols, name):
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[{name}] Missing columns: {missing}")

_require_cols(train_seqs, ["target_id","sequence","temporal_cutoff"], "train_sequences")
_require_cols(valid_seqs, ["target_id","sequence","temporal_cutoff"], "validation_sequences")
_require_cols(test_seqs,  ["target_id","sequence","temporal_cutoff"], "test_sequences")
_require_cols(train_labels, ["ID","resname","resid","x_1","y_1","z_1"], "train_labels")
_require_cols(valid_labels, ["ID","resname","resid","x_1","y_1","z_1"], "validation_labels")
_require_cols(sample_sub, ["ID","resname","resid","x_1","y_1","z_1","x_5","y_5","z_5"], "sample_submission")

# label ID format check (non-fatal)
def _check_id_format(labels_df, name):
    s = labels_df["ID"].astype(str)
    ok = s.str.contains(r".+_\d+$", regex=True).mean()
    print(f"- {name}: ID format like '<target>_<resid>' = {ok*100:.1f}%")

# ---------- Folder inventory ----------
def _count_files(p: Path, suffix=None, max_scan=200000):
    if not p.exists():
        return 0
    n = 0
    if suffix is None:
        for _ in p.iterdir():
            n += 1
        return n
    # bounded scan for big dirs
    for root, _, files in os.walk(p):
        for f in files:
            if f.endswith(suffix):
                n += 1
                if n >= max_scan:
                    return n
    return n

msa_n = _count_files(PATHS["MSA_dir"], suffix=".fasta")
cif_n = _count_files(PATHS["PDB_RNA_dir"], suffix=".cif")

# ---------- Print summary ----------
print("=== STAGE 0: Environment & Data Inventory ===")
print(f"COMP_ROOT: {COMP_ROOT}")
print("")
print("Core tables:")
print(f"- train_sequences      : {train_seqs.shape}")
print(f"- validation_sequences : {valid_seqs.shape}")
print(f"- test_sequences       : {test_seqs.shape}")
print(f"- train_labels         : {train_labels.shape}")
print(f"- validation_labels    : {valid_labels.shape}")
print(f"- sample_submission    : {sample_sub.shape}")
print("")
print("Folders:")
print(f"- MSA dir     : {PATHS['MSA_dir']} | exists={PATHS['MSA_dir'].exists()} | fasta_files~={msa_n}")
print(f"- PDB_RNA dir : {PATHS['PDB_RNA_dir']} | exists={PATHS['PDB_RNA_dir'].exists()} | cif_files~={cif_n}")
print(f"- extra dir   : {PATHS['extra_dir']} | exists={PATHS['extra_dir'].exists()}")
print("")
print("Key extra files (optional):")
for k in ["extra_parse_fasta","extra_rna_metadata","extra_readme","pdb_seqres_fasta","pdb_release_dates"]:
    p = PATHS[k]
    print(f"- {k:17s}: exists={p.exists()} | {p}")

print("")
print("Quick label ID checks:")
_check_id_format(train_labels, "train_labels")
_check_id_format(valid_labels, "validation_labels")

print("")
print("Example rows:")
print("- train_sequences head:")
display(train_seqs.head(2))
print("- sample_submission head:")
display(sample_sub.head(2))


/tmp/ipykernel_24/3371850060.py:40: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  train_labels = pd.read_csv(PATHS["train_labels"])


=== STAGE 0: Environment & Data Inventory ===
COMP_ROOT: /kaggle/input/stanford-rna-3d-folding-2

Core tables:
- train_sequences      : (5716, 8)
- validation_sequences : (28, 8)
- test_sequences       : (28, 8)
- train_labels         : (7794971, 8)
- validation_labels    : (9762, 126)
- sample_submission    : (9762, 18)

Folders:
- MSA dir     : /kaggle/input/stanford-rna-3d-folding-2/MSA | exists=True | fasta_files~=5744
- PDB_RNA dir : /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA | exists=True | cif_files~=9564
- extra dir   : /kaggle/input/stanford-rna-3d-folding-2/extra | exists=True

Key extra files (optional):
- extra_parse_fasta: exists=True | /kaggle/input/stanford-rna-3d-folding-2/extra/parse_fasta_py.py
- extra_rna_metadata: exists=True | /kaggle/input/stanford-rna-3d-folding-2/extra/rna_metadata.csv
- extra_readme     : exists=True | /kaggle/input/stanford-rna-3d-folding-2/extra/README.md
- pdb_seqres_fasta : exists=True | /kaggle/input/stanford-rna-3d-folding-2/PDB_RNA/

,target_id,sequence,temporal_cutoff,description,stoichiometry,all_sequences,ligand_ids,ligand_SMILES
0,4TNA,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1978-04-12,FURTHER REFINEMENT OF THE STRUCTURE OF YEAST T...,A:1,>4TNA_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG,[Mg+2]
1,6TNA,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,1979-01-16,CRYSTAL STRUCTURE OF YEAST PHENYLALANINE T-RNA...,A:1,>6TNA_1|Chain A[auth A]|TRNAPHE|\nGCGGAUUUAGCU...,MG,[Mg+2]


- sample_submission head:


,ID,resname,resid,x_1,y_1,z_1,x_2,y_2,z_2,x_3,y_3,z_3,x_4,y_4,z_4,x_5,y_5,z_5
0,8ZNQ_1,A,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,8ZNQ_2,C,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


# Tamplate Libarary Construction

In [2]:
# ============================================================
# Template Library Construction (ONE CELL)
# - Build template library from PDB_RNA sequences + release dates
# - No DtypeWarning / noisy warnings
# Output globals:
#   df_tpl  : template library (template_id, pdb_id, chain_id, sequence, length, release_date)
#   TPL_PATH: cached parquet path
# ============================================================

import re, warnings
from pathlib import Path
import pandas as pd

# ---- silence common notebook warnings (incl. DtypeWarning) ----
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- paths ----
COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
PDB_DIR   = COMP_ROOT / "PDB_RNA"
FASTA_PDB = PDB_DIR / "pdb_seqres_NA.fasta"
DATES_CSV = PDB_DIR / "pdb_release_dates_NA.csv"

assert COMP_ROOT.exists(), f"Missing COMP_ROOT: {COMP_ROOT}"
assert FASTA_PDB.exists(), f"Missing FASTA: {FASTA_PDB}"
assert DATES_CSV.exists(), f"Missing release dates CSV: {DATES_CSV}"

# ---- load release dates (per PDB entry) ----
df_dates = pd.read_csv(DATES_CSV, dtype=str, low_memory=False)
# robust column naming
cols = [c.lower() for c in df_dates.columns]
df_dates.columns = cols
# try common names
pdb_col = "pdb_id" if "pdb_id" in cols else ("entry_id" if "entry_id" in cols else cols[0])
date_col = "release_date" if "release_date" in cols else ("date" if "date" in cols else cols[1])

df_dates = df_dates[[pdb_col, date_col]].copy()
df_dates.columns = ["pdb_id", "release_date"]
df_dates["pdb_id"] = df_dates["pdb_id"].astype(str).str.upper().str.strip()
df_dates["release_date"] = pd.to_datetime(df_dates["release_date"], errors="coerce")

# ---- parse FASTA (streaming, minimal dependencies) ----
records = []
header = None
seq_chunks = []

def _flush_record(h, chunks):
    if h is None:
        return
    seq = "".join(chunks).strip().upper()
    if not seq:
        return

    # first token is usually something like "1ABC_A" or similar
    token = h.split()[0]
    pdb_id = token[:4].upper()

    # chain id guess: after "_" (e.g., 1ABC_A)
    chain_id = None
    if "_" in token:
        chain_id = token.split("_", 1)[1]
        chain_id = chain_id.strip() if chain_id else None

    # fallback: try "Chain X" or "Chains X" in header
    if not chain_id:
        m = re.search(r"\bChains?\s+([A-Za-z0-9])\b", h)
        if m:
            chain_id = m.group(1)

    template_id = f"{pdb_id}_{chain_id}" if chain_id else pdb_id
    records.append((template_id, pdb_id, chain_id, seq, len(seq), h))

with open(FASTA_PDB, "r", encoding="utf-8", errors="ignore") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        if line.startswith(">"):
            _flush_record(header, seq_chunks)
            header = line[1:].strip()
            seq_chunks = []
        else:
            seq_chunks.append(line)
_flush_record(header, seq_chunks)

df_tpl = pd.DataFrame(records, columns=["template_id","pdb_id","chain_id","sequence","length","header"])
df_tpl = df_tpl.merge(df_dates, on="pdb_id", how="left")
df_tpl = df_tpl.drop(columns=["header"])  # keep minimal

# ---- cache (optional but useful) ----
TPL_PATH = Path("/kaggle/working/template_library.parquet")
df_tpl.to_parquet(TPL_PATH, index=False)

# ---- compact summary ----
print("=== Template Library Construction ===")
print(f"Templates: {len(df_tpl):,} | Unique PDB: {df_tpl['pdb_id'].nunique():,}")
print(f"Length (min/median/max): {df_tpl['length'].min()} / {int(df_tpl['length'].median())} / {df_tpl['length'].max()}")
if df_tpl["release_date"].notna().any():
    print(f"Release dates (min/max): {df_tpl['release_date'].min().date()} -> {df_tpl['release_date'].max().date()}")
else:
    print("Release dates: (missing in merge)")

display(df_tpl.head(3))


=== Template Library Construction ===
Templates: 26,255 | Unique PDB: 9,564
Length (min/median/max): 2 / 74 / 19000
Release dates (min/max): 1978-04-12 -> 2025-12-17


,template_id,pdb_id,chain_id,sequence,length,release_date
0,4TNA_A,4TNA,A,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,76,1978-04-12
1,6TNA_A,6TNA,A,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,76,1979-01-16
2,1TRA_A,1TRA,A,GCGGAUUUAGCUCAGUUGGGAGAGCGCCAGACUGAAGAUCUGGAGG...,76,1986-07-14


# Fast Candidate Retriaeval

In [3]:
# ============================================================
#  Fast Candidate Retrieval (ONE CELL)
# - Build deterministic sampled k-mer inverted index over PDB_RNA template library
# - Provides fast shortlist function for later alignment/ranking stages
# - No noisy warnings
# Requires: df_tpl from STAGE 1 OR /kaggle/working/template_library.parquet
# Outputs globals:
#   TPL (dict)  : template metadata arrays
#   KMER_INDEX  : dict[kmer] -> list of template indices
#   retrieve_candidates(), filter_candidates()
# ============================================================

import warnings, zlib
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- load template library ----
TPL_PATH = Path("/kaggle/working/template_library.parquet")
if "df_tpl" not in globals():
    assert TPL_PATH.exists(), f"Missing {TPL_PATH}. Jalankan STAGE 1 dulu."
    df_tpl = pd.read_parquet(TPL_PATH)

# minimal columns
need = ["template_id","pdb_id","chain_id","sequence","length","release_date"]
missing = [c for c in need if c not in df_tpl.columns]
if missing:
    raise ValueError(f"df_tpl missing columns: {missing}")

# normalize types
df_tpl = df_tpl.copy()
df_tpl["sequence"] = df_tpl["sequence"].astype(str)
df_tpl["length"] = df_tpl["length"].astype(int)
df_tpl["release_date"] = pd.to_datetime(df_tpl["release_date"], errors="coerce")

# ---- config (tune later) ----
K = 6                    # k-mer length
MOD = 4                  # keep k-mers where crc32(kmer) % MOD == 0 (sampling rate ~1/MOD)
MAX_KMERS_PER_SEQ = 1200 # cap for very long sequences (controls build time/memory)
MAX_LEN_FULL = 2000      # sequences <= this use step=1 for k-mer scanning

# ---- build inverted index ----
seqs = df_tpl["sequence"].tolist()
lengths = df_tpl["length"].to_numpy(np.int32)
release_dates = df_tpl["release_date"].to_numpy("datetime64[ns]")
template_ids = df_tpl["template_id"].astype(str).tolist()

KMER_INDEX = {}  # dict[str] -> list[int]
total_postings = 0

for i, s in enumerate(seqs):
    L = len(s)
    if L < K:
        continue

    # stride to limit kmers for long sequences
    if L <= MAX_LEN_FULL:
        step = 1
    else:
        step = max(1, (L - K + 1) // max(1, MAX_KMERS_PER_SEQ))

    # optional per-seq dedup for small sequences (reduces postings)
    use_seen = (step == 1 and L <= 2000)
    seen = set() if use_seen else None

    end = L - K + 1
    for p in range(0, end, step):
        km = s[p:p+K]
        # deterministic sampling to reduce index size
        if (zlib.crc32(km.encode("ascii", "ignore")) % MOD) != 0:
            continue
        if seen is not None:
            if km in seen:
                continue
            seen.add(km)
        lst = KMER_INDEX.get(km)
        if lst is None:
            KMER_INDEX[km] = [i]
        else:
            lst.append(i)
        total_postings += 1

TPL = {
    "template_ids": template_ids,
    "seqs": seqs,
    "lengths": lengths,
    "release_dates": release_dates,  # datetime64[ns], can compare to cutoff
}

# ---- retrieval helpers ----
def retrieve_candidates(query_seq: str, top_k: int = 800):
    """
    Fast shortlist via sampled k-mer voting.
    Returns: list of (tpl_index, votes) sorted by votes desc.
    """
    q = str(query_seq)
    L = len(q)
    if L < K:
        return []

    counts = {}
    end = L - K + 1
    # step=1 for query (short query); sampling already reduces workload
    for p in range(0, end):
        km = q[p:p+K]
        if (zlib.crc32(km.encode("ascii", "ignore")) % MOD) != 0:
            continue
        hits = KMER_INDEX.get(km)
        if not hits:
            continue
        for idx in hits:
            counts[idx] = counts.get(idx, 0) + 1

    if not counts:
        return []

    # partial sort
    items = list(counts.items())
    items.sort(key=lambda x: x[1], reverse=True)
    return items[:top_k]

def filter_candidates(cands, query_len: int, temporal_cutoff=None, len_ratio_tol: float = 0.50):
    """
    Filter shortlist by temporal cutoff and length ratio.
    - temporal_cutoff: 'YYYY-MM-DD' or pd.Timestamp or None
    Returns: filtered list of (tpl_index, votes).
    """
    if not cands:
        return []

    # cutoff
    if temporal_cutoff is None or str(temporal_cutoff) == "nan":
        cutoff_ns = None
    else:
        cutoff_ts = pd.to_datetime(temporal_cutoff, errors="coerce")
        cutoff_ns = np.datetime64(cutoff_ts.to_datetime64()) if pd.notna(cutoff_ts) else None

    out = []
    qL = int(query_len)
    for idx, v in cands:
        tL = int(TPL["lengths"][idx])
        if abs(tL - qL) / max(tL, qL) > len_ratio_tol:
            continue
        if cutoff_ns is not None:
            rd = TPL["release_dates"][idx]
            # allow NaT -> drop (safer)
            if np.isnat(rd) or rd > cutoff_ns:
                continue
        out.append((idx, v))
    return out

# ---- compact summary ----
print("=== Fast Candidate Retrieval ===")
print(f"Templates: {len(seqs):,}")
print(f"K={K} | sampling MOD={MOD} (~{1/MOD:.0%} kept) | index keys: {len(KMER_INDEX):,} | postings: {total_postings:,}")

# ---- quick smoke test (optional, cheap) ----
TEST_SEQS_PATH = Path("/kaggle/input/stanford-rna-3d-folding-2/test_sequences.csv")
if TEST_SEQS_PATH.exists():
    _t = pd.read_csv(TEST_SEQS_PATH, usecols=["target_id","sequence","temporal_cutoff"], dtype=str, low_memory=False).head(1)
    qseq = _t.loc[0, "sequence"]
    cutoff = _t.loc[0, "temporal_cutoff"]
    cand = retrieve_candidates(qseq, top_k=2000)
    cand2 = filter_candidates(cand, len(qseq), temporal_cutoff=cutoff, len_ratio_tol=0.50)[:10]
    print(f"Smoke test on target={_t.loc[0,'target_id']} | query_len={len(qseq)} | candidates(after filter)={len(cand2)}")
    for idx, v in cand2[:5]:
        print("  ", TPL["template_ids"][idx], "votes=", v, "len=", int(TPL["lengths"][idx]))


=== Fast Candidate Retrieval ===
Templates: 26,255
K=6 | sampling MOD=4 (~25% kept) | index keys: 1,537 | postings: 1,989,285
Smoke test on target=8ZNQ | query_len=30 | candidates(after filter)=10
   8ZNQ_A votes= 5 len= 30
   2O32_A votes= 3 len= 20
   2O33_A votes= 2 len= 20
   6MUT_H votes= 2 len= 45
   4V5Z_AE votes= 2 len= 32


# MSA-Expanded Querying

In [4]:
# ============================================================
# MSA-Expanded Querying (ONE CELL)
# - Use MSA/{target_id}.MSA.fasta to expand queries with homolog sequences
# - For each query (target + homologs): run fast k-mer retrieval (STAGE 3)
# - Aggregate votes across queries -> stronger shortlist
# - No noisy warnings
#
# Requires from STAGE 3:
#   - TPL, KMER_INDEX, retrieve_candidates(), filter_candidates()
#
# Outputs globals:
#   - read_msa_queries()
#   - msa_expand_candidates()
# ============================================================

import warnings, re
from pathlib import Path
import pandas as pd

warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- require STAGE 3 objects ----
for need in ["TPL", "KMER_INDEX", "retrieve_candidates", "filter_candidates"]:
    if need not in globals():
        raise RuntimeError(f"Missing '{need}'. Jalankan STAGE 3 dulu.")

COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
MSA_DIR = COMP_ROOT / "MSA"
assert MSA_DIR.exists(), f"Missing MSA dir: {MSA_DIR}"

# ---- helpers ----
_ACGU = set("ACGU")

def _clean_seq(s: str) -> str:
    # remove gaps and keep only canonical RNA bases
    s = s.upper().replace("-", "")
    s = "".join([c for c in s if c in _ACGU])
    return s

def read_msa_queries(target_id: str, base_sequence: str, max_homologs: int = 20):
    """
    Read MSA fasta and return query list: [base_sequence] + homolog sequences (cleaned).
    Safe fallback: if MSA file missing -> returns [base_sequence] only.
    """
    queries = []
    base = _clean_seq(str(base_sequence))
    if base:
        queries.append(base)

    msa_path = MSA_DIR / f"{target_id}.MSA.fasta"
    if not msa_path.exists():
        return queries

    seqs = []
    cur = []
    with open(msa_path, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(">"):
                if cur:
                    seqs.append(_clean_seq("".join(cur)))
                    cur = []
            else:
                cur.append(line)
        if cur:
            seqs.append(_clean_seq("".join(cur)))

    # keep only useful, dedup, and cap
    seen = set(queries)
    for s in seqs:
        if not s:
            continue
        # keep length reasonably close to base (avoid MSA rows dominated by gaps)
        if base and (abs(len(s) - len(base)) / max(len(s), len(base)) > 0.50):
            continue
        if s not in seen:
            queries.append(s)
            seen.add(s)
        if len(queries) >= 1 + max_homologs:
            break

    return queries

def msa_expand_candidates(
    target_id: str,
    sequence: str,
    temporal_cutoff=None,
    max_homologs: int = 20,
    top_k_per_query: int = 800,
    final_top_k: int = 1200,
    len_ratio_tol: float = 0.50,
):
    """
    Returns expanded shortlist: list of (tpl_index, agg_score) sorted desc.
    - agg_score = sum of votes across queries (base + homologs)
    """
    queries = read_msa_queries(target_id, sequence, max_homologs=max_homologs)
    if not queries:
        return []

    agg = {}
    for q in queries:
        c = retrieve_candidates(q, top_k=top_k_per_query)
        c = filter_candidates(c, query_len=len(q), temporal_cutoff=temporal_cutoff, len_ratio_tol=len_ratio_tol)
        for idx, v in c:
            agg[idx] = agg.get(idx, 0) + int(v)

    if not agg:
        return []

    items = list(agg.items())
    items.sort(key=lambda x: x[1], reverse=True)
    return items[:final_top_k]

# ---- quick smoke test (cheap) ----
TEST_SEQS_PATH = COMP_ROOT / "test_sequences.csv"
if TEST_SEQS_PATH.exists():
    _t = pd.read_csv(TEST_SEQS_PATH, usecols=["target_id","sequence","temporal_cutoff"], dtype=str, low_memory=False).head(1)
    tid = _t.loc[0, "target_id"]
    seq = _t.loc[0, "sequence"]
    cutoff = _t.loc[0, "temporal_cutoff"]

    base_cand = filter_candidates(retrieve_candidates(seq, top_k=800), len(seq), temporal_cutoff=cutoff)[:10]
    msa_cand  = msa_expand_candidates(tid, seq, temporal_cutoff=cutoff, max_homologs=20, top_k_per_query=800, final_top_k=1200)[:10]

    print("=== MSA-Expanded Querying ===")
    print(f"Target: {tid} | len={len(seq)} | MSA file exists={ (MSA_DIR / f'{tid}.MSA.fasta').exists() }")
    print(f"Queries used (base+homologs): {len(read_msa_queries(tid, seq, max_homologs=20))}")
    print("Top-5 BASE candidates:")
    for idx, v in base_cand[:5]:
        print("  ", TPL["template_ids"][idx], "score=", v, "len=", int(TPL["lengths"][idx]))
    print("Top-5 MSA-EXPANDED candidates:")
    for idx, v in msa_cand[:5]:
        print("  ", TPL["template_ids"][idx], "score=", v, "len=", int(TPL["lengths"][idx]))


=== MSA-Expanded Querying ===
Target: 8ZNQ | len=30 | MSA file exists=True
Queries used (base+homologs): 14
Top-5 BASE candidates:
   8ZNQ_A score= 5 len= 30
   2O32_A score= 3 len= 20
   2O33_A score= 2 len= 20
   6MUT_H score= 2 len= 45
Top-5 MSA-EXPANDED candidates:
   2O32_A score= 17 len= 20
   8ZNQ_A score= 14 len= 30
   2O33_A score= 4 len= 20
   6MUT_H score= 2 len= 45
   8WCE_T score= 2 len= 31


# High-Precision Alignment & Ranking 

In [5]:
# ============================================================
# High-Precision Alignment & Ranking (ONE CELL)
# - Take shortlist from MSA-expanded retrieval (STAGE 4)
# - Run precise pairwise alignment on top candidates only
# - Compute alignment metrics (score, coverage, identity, gaps) + final rank score
#
# Requires from STAGE 4:
#   - TPL (template_ids, seqs, lengths, release_dates)
#   - msa_expand_candidates()
#
# Outputs globals:
#   - align_and_rank_templates()
#   - pick_top_templates()
# ============================================================

import warnings, math
from pathlib import Path
import numpy as np
import pandas as pd
from Bio import pairwise2
from Bio import BiopythonDeprecationWarning
import warnings

warnings.filterwarnings("ignore", category=BiopythonDeprecationWarning)
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- require previous stages ----
for need in ["TPL", "msa_expand_candidates"]:
    if need not in globals():
        raise RuntimeError(f"Missing '{need}'. Jalankan STAGE 3-4 dulu.")

def _safe_to_datetime64(x):
    ts = pd.to_datetime(x, errors="coerce")
    if pd.isna(ts):
        return None
    return np.datetime64(ts.to_datetime64())

def _alignment_metrics(aligned_q: str, aligned_t: str, raw_score: float, q_len: int, t_len: int):
    """
    Compute robust alignment metrics from aligned strings.
    """
    assert len(aligned_q) == len(aligned_t)
    L = len(aligned_q)

    # aligned pairs where both are not gaps
    aligned_pairs = 0
    matches = 0
    gap_q = 0
    gap_t = 0

    for a, b in zip(aligned_q, aligned_t):
        if a == "-":
            gap_q += 1
            continue
        if b == "-":
            gap_t += 1
            continue
        aligned_pairs += 1
        if a == b:
            matches += 1

    coverage_q = aligned_pairs / max(1, q_len)           # fraction of query residues aligned to template residues
    identity  = matches / max(1, aligned_pairs)          # among aligned pairs
    gap_frac  = (gap_q + gap_t) / max(1, L)

    # normalize score similar to your baseline: divide by best possible per residue (match=2)
    sim = raw_score / (2.0 * min(q_len, t_len) + 1e-9)

    return {
        "sim": float(sim),
        "coverage_q": float(coverage_q),
        "identity": float(identity),
        "gap_frac": float(gap_frac),
        "aligned_pairs": int(aligned_pairs),
        "matches": int(matches),
        "aln_len": int(L),
    }

def align_and_rank_templates(
    target_id: str,
    sequence: str,
    temporal_cutoff=None,
    max_homologs: int = 20,
    shortlist_k_per_query: int = 800,
    shortlist_final_k: int = 1200,
    align_top: int = 40,
    len_ratio_tol: float = 0.50,
):
    """
    1) Get MSA-expanded shortlist (tpl_idx, agg_votes)
    2) Align top-N (by agg_votes) using global alignment
    3) Rank with a composite score prioritizing:
       - sim (normalized alignment score)
       - coverage on query
       - identity
       - small penalty for gap_frac
       - mild boost from retrieval votes
    Returns DataFrame sorted by rank_score desc with aligned strings included.
    """
    q = str(sequence).upper()
    q_len = len(q)

    # shortlist from STAGE 4
    cands = msa_expand_candidates(
        target_id=target_id,
        sequence=q,
        temporal_cutoff=temporal_cutoff,
        max_homologs=max_homologs,
        top_k_per_query=shortlist_k_per_query,
        final_top_k=shortlist_final_k,
        len_ratio_tol=len_ratio_tol,
    )
    if not cands:
        return pd.DataFrame(columns=[
            "template_id","tpl_index","votes","sim","coverage_q","identity","gap_frac","rank_score",
            "aligned_q","aligned_t","template_len","release_date"
        ])

    # keep only top for alignment (fast)
    cands = cands[:max(align_top, 1)]

    # temporal cutoff (defensive; already filtered earlier)
    cutoff_ns = _safe_to_datetime64(temporal_cutoff) if temporal_cutoff is not None else None

    rows = []
    for tpl_idx, votes in cands:
        t_seq = TPL["seqs"][tpl_idx].upper()
        t_len = int(TPL["lengths"][tpl_idx])
        t_id  = TPL["template_ids"][tpl_idx]
        rd    = TPL["release_dates"][tpl_idx]

        if cutoff_ns is not None:
            if np.isnat(rd) or rd > cutoff_ns:
                continue

        # precise alignment (same scoring as baseline spirit)
        aln = pairwise2.align.globalms(q, t_seq, 2, -1, -10, -0.5, one_alignment_only=True)
        if not aln:
            continue
        a = aln[0]
        aligned_q = a.seqA
        aligned_t = a.seqB
        raw_score = float(a.score)

        m = _alignment_metrics(aligned_q, aligned_t, raw_score, q_len=q_len, t_len=t_len)

        # composite rank score (simple, stable)
        # - sim is dominant
        # - coverage helps avoid partial/local matches
        # - identity helps avoid weird alignments
        # - gap penalty discourages excessive gaps
        # - votes give small prior from retrieval
        vote_boost = math.log1p(votes) / 10.0  # mild
        rank_score = (
            0.60 * m["sim"] +
            0.25 * m["coverage_q"] +
            0.15 * m["identity"] -
            0.10 * m["gap_frac"] +
            0.05 * vote_boost
        )

        rows.append({
            "template_id": t_id,
            "tpl_index": int(tpl_idx),
            "votes": int(votes),
            "sim": m["sim"],
            "coverage_q": m["coverage_q"],
            "identity": m["identity"],
            "gap_frac": m["gap_frac"],
            "rank_score": float(rank_score),
            "aligned_q": aligned_q,
            "aligned_t": aligned_t,
            "template_len": t_len,
            "release_date": pd.to_datetime(rd) if not np.isnat(rd) else pd.NaT,
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    df = df.sort_values(["rank_score","sim","coverage_q","identity","votes"], ascending=False).reset_index(drop=True)
    return df

def pick_top_templates(df_ranked: pd.DataFrame, top_n: int = 10):
    """
    Return a compact list of top templates to feed the next stages.
    """
    if df_ranked is None or len(df_ranked) == 0:
        return []
    out = []
    for _, r in df_ranked.head(top_n).iterrows():
        out.append({
            "template_id": r["template_id"],
            "tpl_index": int(r["tpl_index"]),
            "rank_score": float(r["rank_score"]),
            "sim": float(r["sim"]),
            "coverage_q": float(r["coverage_q"]),
            "identity": float(r["identity"]),
            "votes": int(r["votes"]),
            "aligned_q": r["aligned_q"],
            "aligned_t": r["aligned_t"],
        })
    return out

# ---- smoke test ----
COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
TEST_SEQS_PATH = COMP_ROOT / "test_sequences.csv"
if TEST_SEQS_PATH.exists():
    _t = pd.read_csv(TEST_SEQS_PATH, usecols=["target_id","sequence","temporal_cutoff"], dtype=str, low_memory=False).head(1)
    tid = _t.loc[0, "target_id"]
    seq = _t.loc[0, "sequence"]
    cutoff = _t.loc[0, "temporal_cutoff"]

    df_rank = align_and_rank_templates(
        target_id=tid, sequence=seq, temporal_cutoff=cutoff,
        max_homologs=20, shortlist_k_per_query=800, shortlist_final_k=1200, align_top=40
    )

    print("=== High-Precision Alignment & Ranking ===")
    print(f"Target: {tid} | len={len(seq)} | ranked_templates={len(df_rank)}")
    if len(df_rank):
        display(df_rank[["template_id","votes","sim","coverage_q","identity","gap_frac","rank_score","template_len","release_date"]].head(10))


=== High-Precision Alignment & Ranking ===
Target: 8ZNQ | len=30 | ranked_templates=39


/usr/local/lib/python3.12/dist-packages/Bio/pairwise2.py:278: BiopythonDeprecationWarning: Bio.pairwise2 has been deprecated, and we intend to remove it in a future release of Biopython. As an alternative, please consider using Bio.Align.PairwiseAligner as a replacement, and contact the Biopython developers if you still need the Bio.pairwise2 module.
  warnings.warn(


,template_id,votes,sim,coverage_q,identity,gap_frac,rank_score,template_len,release_date
0,8ZNQ_A,14,1.000000,1.000000,1.00,0.000000,1.013540,30,2025-06-04
1,6MUR_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2018-12-12
2,6MUS_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2018-12-12
3,6MUT_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2018-12-12
4,6MUU_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2018-12-19
5,6O7E_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2019-07-31
6,6O7H_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2019-07-31
7,6O7I_G,1,0.025000,1.000000,0.50,0.210526,0.322413,38,2019-07-31
8,2O32_A,17,0.025000,0.666667,0.75,0.333333,0.275285,20,2007-02-06
9,8WCE_T,2,-0.066667,1.000000,0.40,0.032258,0.272267,31,2024-05-22


# Coordinate Extraction with Caching

In [6]:
# ============================================================
# Coordinate Extraction with Caching (ONE CELL) [REVISI FULL v4]
# FIX UTAMA:
# - CIF filename di PDB_RNA cenderung lowercase (mis. 8znq.cif), sedangkan pdb_id di tabel uppercase.
# - Sekarang resolve path secara case-insensitive: coba lower/upper/as-is + fallback glob.
#
# Fitur tetap:
# 1) try exact chain match (auth_chains True/False)
# 2) fallback: scan semua chain, pilih yang C1' count paling dekat expected_len
# Cache: /kaggle/working/c1_cache/{pdb_id}_{chain_id}.npz
# No warning spam
# ============================================================

import warnings
from pathlib import Path
import numpy as np
import pandas as pd

# ---- silence warnings ----
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
try:
    from Bio import BiopythonDeprecationWarning, BiopythonWarning
    warnings.filterwarnings("ignore", category=BiopythonDeprecationWarning)
    warnings.filterwarnings("ignore", category=BiopythonWarning)
except Exception:
    pass

from Bio.PDB.MMCIFParser import MMCIFParser

# ---- paths ----
COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
PDB_DIR   = COMP_ROOT / "PDB_RNA"
assert PDB_DIR.exists(), f"Missing PDB_RNA dir: {PDB_DIR}"

C1_CACHE = Path("/kaggle/working/c1_cache")
C1_CACHE.mkdir(parents=True, exist_ok=True)

# ---- load df_tpl ----
TPL_PATH = Path("/kaggle/working/template_library.parquet")
if "df_tpl" not in globals():
    if not TPL_PATH.exists():
        raise RuntimeError("template_library.parquet missing. Jalankan STAGE 1 dulu.")
    df_tpl = pd.read_parquet(TPL_PATH)

_tpl_lookup = df_tpl.set_index("template_id")[["pdb_id","chain_id","length"]].to_dict("index")

def _resolve_cif_path(pdb_id: str):
    """Return existing cif path for pdb_id with case-insensitive handling."""
    p = str(pdb_id).strip()
    cands = [
        PDB_DIR / f"{p}.cif",
        PDB_DIR / f"{p.lower()}.cif",
        PDB_DIR / f"{p.upper()}.cif",
    ]
    for cp in cands:
        if cp.exists():
            return cp
    # fallback glob (case-insensitive-ish by checking both)
    gl1 = list(PDB_DIR.glob(f"{p.lower()}.cif"))
    if gl1:
        return gl1[0]
    gl2 = list(PDB_DIR.glob(f"{p.upper()}.cif"))
    if gl2:
        return gl2[0]
    return None

def _extract_c1_from_chain(chain):
    coords, resseq = [], []
    for res in chain.get_residues():
        atom = None
        if "C1'" in res:
            atom = res["C1'"]
        elif "C1*" in res:
            atom = res["C1*"]
        else:
            continue

        xyz = atom.get_coord()
        if xyz is None or len(xyz) != 3:
            continue

        try:
            rnum = int(res.id[1])
        except Exception:
            rnum = -1

        coords.append([float(xyz[0]), float(xyz[1]), float(xyz[2])])
        resseq.append(rnum)

    if not coords:
        return None

    # sort by residue number if possible
    if any(r >= 0 for r in resseq):
        order = np.argsort([r if r >= 0 else 10**9 for r in resseq])
        coords = np.asarray(coords, dtype=np.float32)[order]
        resseq = np.asarray(resseq, dtype=np.int32)[order]
    else:
        coords = np.asarray(coords, dtype=np.float32)
        resseq = np.asarray(resseq, dtype=np.int32)

    return {"coords": coords, "resseq": resseq}

def _parse_first_model(cif_path: Path, pdb_id: str, auth_chains: bool):
    try:
        parser = MMCIFParser(QUIET=True, auth_chains=auth_chains)
        st = parser.get_structure(pdb_id, str(cif_path))
        model = next(st.get_models(), None)
        return model
    except Exception:
        return None

def get_c1_coords(pdb_id: str, chain_id: str, expected_len=None, use_cache: bool = True, debug: bool = False):
    pdb = str(pdb_id).upper().strip()
    ch  = None if chain_id is None or str(chain_id) == "nan" else str(chain_id).strip()

    cif_path = _resolve_cif_path(pdb_id)
    if cif_path is None or (not cif_path.exists()):
        if debug:
            print(f"DEBUG: CIF NOT FOUND for pdb_id={pdb_id} (looked for .cif variants)")
        return None

    cache_path = C1_CACHE / f"{pdb}_{ch}.npz"
    if use_cache and cache_path.exists():
        try:
            d = np.load(cache_path, allow_pickle=False)
            return {
                "pdb_id": pdb,
                "chain_id": ch,
                "chosen_chain_id": str(d["chosen_chain_id"]) if "chosen_chain_id" in d else ch,
                "coords": d["coords"].astype(np.float32),
                "resseq": d["resseq"].astype(np.int32),
            }
        except Exception:
            pass

    expL = None
    if expected_len is not None and str(expected_len) != "nan":
        try:
            expL = int(expected_len)
        except Exception:
            expL = None

    best = None  # (score, chosen_chain_id, coords_dict)
    tried_models = 0

    for auth_chains in (True, False):
        model = _parse_first_model(cif_path, pdb, auth_chains=auth_chains)
        if model is None:
            continue
        tried_models += 1

        chains = list(model.get_chains())
        if not chains:
            continue

        # (1) exact match
        if ch:
            for c in chains:
                if str(c.id).strip() == ch:
                    ex = _extract_c1_from_chain(c)
                    if ex is not None:
                        best = (0, str(c.id).strip(), ex)
                        break
            if best is not None and best[0] == 0:
                break

        # (2) fallback: choose chain by closeness to expected length (or max C1 count)
        for c in chains:
            ex = _extract_c1_from_chain(c)
            if ex is None:
                continue
            n = int(ex["coords"].shape[0])
            score = abs(n - expL) if expL is not None else -n
            cand = (score, str(c.id).strip(), ex)
            if best is None:
                best = cand
            else:
                if cand[0] < best[0] or (cand[0] == best[0] and n > int(best[2]["coords"].shape[0])):
                    best = cand

    if best is None:
        if debug:
            print(f"DEBUG: No C1' found in CIF={cif_path.name} | tried_models={tried_models} | requested_chain={ch} | expected_len={expL}")
        return None

    _, chosen_chain_id, ex = best
    out = {
        "pdb_id": pdb,
        "chain_id": ch,
        "chosen_chain_id": chosen_chain_id,
        "coords": ex["coords"],
        "resseq": ex["resseq"],
    }

    try:
        np.savez_compressed(
            cache_path,
            coords=out["coords"],
            resseq=out["resseq"],
            chosen_chain_id=np.array(chosen_chain_id),
            cif_name=np.array(str(cif_path.name)),
        )
    except Exception:
        pass

    if debug:
        print(f"DEBUG: CIF used={cif_path.name} | requested_chain={ch} | chosen_chain={chosen_chain_id} | c1_count={len(out['coords'])} | expected_len={expL}")

    return out

def get_c1_for_template(template_id: str, use_cache: bool = True, debug: bool = False):
    info = _tpl_lookup.get(str(template_id))
    if info is None:
        return None
    return get_c1_coords(
        info["pdb_id"],
        info["chain_id"],
        expected_len=info.get("length", None),
        use_cache=use_cache,
        debug=debug
    )

# ---- smoke test ----
print("===  Coordinate Extraction with Caching (REVISI FULL v4) ===")
_smoke_tpl = None
if "df_rank" in globals() and isinstance(globals()["df_rank"], pd.DataFrame) and len(globals()["df_rank"]) > 0:
    _smoke_tpl = globals()["df_rank"].iloc[0]["template_id"]
else:
    _smoke_tpl = df_tpl.iloc[0]["template_id"] if len(df_tpl) else None

if _smoke_tpl is None:
    print("Smoke test skipped (no template).")
else:
    r = get_c1_for_template(_smoke_tpl, use_cache=False, debug=True)  # use_cache=False biar bener-bener ngetes extract
    if r is None:
        print(f"Smoke test: FAILED for template={_smoke_tpl}")
    else:
        print(f"Smoke test: OK template={_smoke_tpl} | C1' atoms={len(r['coords'])} | chosen_chain={r['chosen_chain_id']} | cache={C1_CACHE}")
        print("First 3 coords:", r["coords"][:3])


===  Coordinate Extraction with Caching (REVISI FULL v4) ===
DEBUG: CIF used=8znq.cif | requested_chain=A | chosen_chain=A | c1_count=30 | expected_len=30
Smoke test: OK template=8ZNQ_A | C1' atoms=30 | chosen_chain=A | cache=/kaggle/working/c1_cache
First 3 coords: [[ -2.054 -15.062  20.736]
 [ -1.971 -15.076  15.338]
 [ -3.35  -13.497  10.444]]


# Tamplate to Target Coordinate Mapping 

In [7]:
# ============================================================
# Template to Target Coordinate Mapping (ONE CELL)
# - Map template C1' coords -> target sequence coords using alignment strings from STAGE 5
# - Fill gaps (template gaps / missing mapped residues) via interpolation/extrapolation
# - Quiet: no warning spam
#
# Requires:
#   - df_rank (from STAGE 5) for a target (has: template_id, aligned_q, aligned_t)
#   - get_c1_for_template() (from STAGE 6)
#
# Outputs globals:
#   - map_coords_via_alignment(query_seq, template_coords, aligned_q, aligned_t)
#   - mapped_coords_from_best(df_rank, query_seq)
# ============================================================

import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- require previous stages ----
for need in ["get_c1_for_template"]:
    if need not in globals():
        raise RuntimeError(f"Missing '{need}'. Jalankan STAGE 6 dulu.")
if "df_rank" not in globals() or not isinstance(df_rank, pd.DataFrame) or len(df_rank) == 0:
    raise RuntimeError("Missing df_rank. Jalankan STAGE 5 (smoke test) dulu untuk suatu target.")

def _fill_nans(coords: np.ndarray):
    """Fill NaNs in (N,3) using linear interpolation, then extrapolate ends."""
    N = coords.shape[0]
    ok = ~np.isnan(coords[:, 0])
    if ok.sum() == 0:
        return None

    # interpolate internal gaps per dimension
    idx = np.arange(N)
    for d in range(3):
        v = coords[:, d]
        okd = ~np.isnan(v)
        if okd.sum() >= 2:
            v[~okd] = np.interp(idx[~okd], idx[okd], v[okd])
        elif okd.sum() == 1:
            v[~okd] = v[okd][0]
        coords[:, d] = v

    # extrapolate leading/trailing if any still NaN (rare after interp)
    ok = ~np.isnan(coords[:, 0])
    if ok.sum() >= 2:
        first = int(np.where(ok)[0][0])
        last  = int(np.where(ok)[0][-1])

        # leading
        if first > 0:
            step = coords[first+1] - coords[first] if first+1 <= last else np.array([0.0, 0.0, 4.0])
            for i in range(first-1, -1, -1):
                coords[i] = coords[i+1] - step

        # trailing
        if last < N-1:
            step = coords[last] - coords[last-1] if last-1 >= first else np.array([0.0, 0.0, 4.0])
            for i in range(last+1, N):
                coords[i] = coords[i-1] + step

    return coords

def _fallback_helix(n: int):
    """Simple helix fallback if nothing can be mapped."""
    coords = np.zeros((n, 3), dtype=np.float32)
    r = 10.0
    rise = 2.5
    ang = 0.6
    for i in range(n):
        a = i * ang
        coords[i] = [r * np.cos(a), r * np.sin(a), i * rise]
    return coords

def map_coords_via_alignment(query_seq: str, template_coords: np.ndarray, aligned_q: str, aligned_t: str):
    """
    Map template coords to query indices using aligned strings.
    - template_coords assumed ordered along template sequence (length ~= template_len).
    """
    q = str(query_seq)
    q_len = len(q)
    out = np.full((q_len, 3), np.nan, dtype=np.float32)

    qi = 0
    ti = 0
    tN = int(template_coords.shape[0])

    L = min(len(aligned_q), len(aligned_t))
    for k in range(L):
        aq = aligned_q[k]
        at = aligned_t[k]

        if aq != "-" and at != "-":
            if qi < q_len and ti < tN:
                out[qi] = template_coords[ti]
            qi += 1
            ti += 1
        elif aq != "-" and at == "-":
            # query residue not present in template (gap) -> keep NaN
            qi += 1
        elif aq == "-" and at != "-":
            ti += 1
        else:
            # both gaps (shouldn't happen often)
            pass

        if qi >= q_len and ti >= tN:
            break

    filled = _fill_nans(out)
    if filled is None:
        return _fallback_helix(q_len)
    return filled.astype(np.float32)

def mapped_coords_from_best(df_rank: pd.DataFrame, query_seq: str, use_cache: bool = True):
    """Use top-1 template in df_rank to produce mapped coords for query."""
    r0 = df_rank.iloc[0]
    tpl_id = r0["template_id"]
    aligned_q = r0["aligned_q"]
    aligned_t = r0["aligned_t"]

    tpl = get_c1_for_template(tpl_id, use_cache=use_cache, debug=False)
    if tpl is None:
        return None

    coords = tpl["coords"]
    mapped = map_coords_via_alignment(query_seq, coords, aligned_q, aligned_t)
    return {"template_id": tpl_id, "coords": mapped}

# ---- smoke test using same target as STAGE 5 smoke (df_rank is for that target) ----
COMP_ROOT = "/kaggle/input/stanford-rna-3d-folding-2"
_test = pd.read_csv(f"{COMP_ROOT}/test_sequences.csv", usecols=["target_id","sequence"], dtype=str, low_memory=False).head(1)
_tid = _test.loc[0, "target_id"]
_seq = _test.loc[0, "sequence"]

res = mapped_coords_from_best(df_rank, _seq, use_cache=True)
print("=== Template to Target Coordinate Mapping ===")
if res is None:
    print(f"FAILED mapping for target={_tid} (template coords missing)")
else:
    mc = res["coords"]
    print(f"Target: {_tid} | len={len(_seq)} | best_template={res['template_id']} | mapped_shape={mc.shape} | nan_any={np.isnan(mc).any()}")
    print("First 10 mapped coords:", mc[:10])


=== Template to Target Coordinate Mapping ===
Target: 8ZNQ | len=30 | best_template=8ZNQ_A | mapped_shape=(30, 3) | nan_any=False
First 10 mapped coords: [[ -2.054 -15.062  20.736]
 [ -1.971 -15.076  15.338]
 [ -3.35  -13.497  10.444]
 [ -5.443 -11.044   6.605]
 [ -6.35   -6.269   4.55 ]
 [ -6.926  -0.899   3.251]
 [ -4.474   4.139  -0.941]
 [ -5.416   9.797   3.376]
 [ -0.388  10.58    0.104]
 [  3.898   9.969  -2.049]]


# Light Geometry Refinement

In [8]:
# ============================================================
#  Light Geometry Refinement (ONE CELL)
# - Gentle smoothing + backbone distance regularization + local clash relief
# - Designed to be safe (won't wildly distort a good template)
# - Quiet: suppress warnings
#
# Requires (from previous stages):
#   - map_coords_via_alignment(...) (STAGE 7) OR a coords array already available
#   - (optional) res, _seq from STAGE 7 smoke test
#
# Outputs globals:
#   - light_refine_coords(coords, seq, ...)
# ============================================================

import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

def _safe_norm(v, eps=1e-12):
    return float(np.sqrt((v*v).sum()) + eps)

def light_refine_coords(
    coords: np.ndarray,
    seq: str,
    n_iter: int = 3,
    smooth_w: float = 0.12,
    dist_min: float = 5.4,
    dist_max: float = 6.6,
    dist_target: float = 6.0,
    dist_strength: float = 0.35,
    min_nonseq: float = 3.8,
    clash_strength: float = 0.25,
    clash_window: int = 40,
):
    """
    coords: (N,3) float
    seq: string, only used for length sanity
    Returns refined (N,3) float32
    """
    x = np.asarray(coords, dtype=np.float32).copy()
    n = x.shape[0]
    if n == 0:
        return x
    if seq is not None and len(seq) != n:
        # do not crash; trust coords length
        pass

    for _ in range(int(max(1, n_iter))):
        # (1) very light smoothing (internal points only)
        if n >= 3 and smooth_w > 0:
            y = x.copy()
            y[1:-1] = (1.0 - smooth_w) * x[1:-1] + (smooth_w * 0.5) * (x[:-2] + x[2:])
            x = y

        # (2) backbone distance regularization (symmetric adjustment)
        if n >= 2 and dist_strength > 0:
            for i in range(n - 1):
                v = x[i+1] - x[i]
                d = _safe_norm(v)
                if d < dist_min or d > dist_max:
                    dirv = v / d
                    delta = (dist_target - d) * dist_strength
                    shift = 0.5 * delta * dirv
                    x[i]   -= shift
                    x[i+1] += shift

        # (3) local clash relief (windowed, non-consecutive only)
        if n >= 4 and clash_strength > 0 and min_nonseq > 0:
            W = int(max(6, clash_window))
            min2 = float(min_nonseq * min_nonseq)
            for i in range(n - 3):
                pi = x[i]
                j0 = i + 2
                j1 = min(n, i + W)
                if j0 >= j1:
                    continue
                pj = x[j0:j1]
                d2 = ((pj - pi) ** 2).sum(axis=1)
                bad = np.where(d2 < min2)[0]
                for b in bad:
                    j = j0 + int(b)
                    v = x[j] - x[i]
                    d = _safe_norm(v)
                    if d <= 1e-6:
                        # random tiny direction
                        v = np.array([1.0, 0.0, 0.0], dtype=np.float32)
                        d = 1.0
                    dirv = v / d
                    need = (min_nonseq - d) * clash_strength
                    shift = 0.5 * need * dirv
                    x[i] -= shift
                    x[j] += shift

    return x.astype(np.float32)

# ---- smoke test (uses Stage 7 variables if present) ----
print("=== Light Geometry Refinement ===")
coords_in = None
seq_in = None
tpl_in = None

if "res" in globals() and isinstance(res, dict) and "coords" in res:
    coords_in = res["coords"]
    tpl_in = res.get("template_id", None)
if "_seq" in globals():
    seq_in = _seq

# fallback: if no coords_in, try build from df_rank + test_sequences first row (same as Stage 7 smoke)
if coords_in is None:
    if "df_rank" in globals() and "map_coords_via_alignment" in globals():
        COMP_ROOT = "/kaggle/input/stanford-rna-3d-folding-2"
        _t = pd.read_csv(f"{COMP_ROOT}/test_sequences.csv", usecols=["target_id","sequence"], dtype=str, low_memory=False).head(1)
        _tid = _t.loc[0, "target_id"]
        seq_in = _t.loc[0, "sequence"]
        r0 = df_rank.iloc[0]
        tpl_in = r0["template_id"]
        tpl = get_c1_for_template(tpl_in, use_cache=True, debug=False)
        coords_in = map_coords_via_alignment(seq_in, tpl["coords"], r0["aligned_q"], r0["aligned_t"])
    else:
        raise RuntimeError("No input coords found. Jalankan STAGE 7 dulu (mapping).")

coords_ref = light_refine_coords(coords_in, seq_in)

# quick metrics
def _seq_stats(c):
    if c.shape[0] < 2:
        return (np.nan, np.nan, np.nan)
    d = np.sqrt(((c[1:] - c[:-1])**2).sum(axis=1))
    return float(d.min()), float(np.median(d)), float(d.max())

mn0, med0, mx0 = _seq_stats(np.asarray(coords_in, dtype=np.float32))
mn1, med1, mx1 = _seq_stats(coords_ref)

print(f"Input  : template={tpl_in} | N={coords_in.shape[0]} | seq_dist(min/med/max)={mn0:.3f}/{med0:.3f}/{mx0:.3f}")
print(f"Refined: template={tpl_in} | N={coords_ref.shape[0]} | seq_dist(min/med/max)={mn1:.3f}/{med1:.3f}/{mx1:.3f}")
print("First 10 refined coords:", coords_ref[:10])


=== Light Geometry Refinement ===
Input  : template=8ZNQ_A | N=30 | seq_dist(min/med/max)=4.835/5.521/8.400
Refined: template=8ZNQ_A | N=30 | seq_dist(min/med/max)=5.034/5.369/6.343
First 10 refined coords: [[ -2.0517056  -15.067855    20.972895  ]
 [ -2.1118503  -14.926949    15.583823  ]
 [ -3.3674068  -13.487185    10.659041  ]
 [ -5.3769646  -10.768459     6.68857   ]
 [ -6.328619    -5.974766     4.541966  ]
 [ -6.485151    -0.72520185   2.8092644 ]
 [ -4.9453735    4.066355     0.26212847]
 [ -4.8414264    9.047952     2.497195  ]
 [ -0.55793154  10.461221     0.26288253]
 [  3.977025     9.770266    -2.1301382 ]]


# Best of 5 Strategy

In [9]:
# ============================================================
# STAGE 9 — Best-of-5 Strategy (ONE CELL)
# - Build 5 diverse predictions from best template-mapped coords:
#   1) base refined
#   2) gentle noise
#   3) slightly stronger noise
#   4) random rigid transform + gentle noise
#   5) different rigid transform + moderate noise
# - Each prediction is lightly refined (STAGE 8)
# - Quiet warnings
#
# Requires:
#   - mapped_coords_from_best (STAGE 7) OR res (dict with coords + template_id)
#   - light_refine_coords (STAGE 8)
#
# Outputs globals:
#   - make_5_predictions(coords_base, seq, seed=0) -> list of 5 (N,3)
#   - preds5 (from smoke test)
# ============================================================

import warnings, numpy as np, pandas as pd
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ---- require previous stage functions ----
for need in ["light_refine_coords"]:
    if need not in globals():
        raise RuntimeError(f"Missing '{need}'. Jalankan STAGE 8 dulu.")

def _rand_rotation(rng):
    # random unit quaternion -> rotation matrix
    u1, u2, u3 = rng.random(3)
    q1 = np.sqrt(1-u1) * np.sin(2*np.pi*u2)
    q2 = np.sqrt(1-u1) * np.cos(2*np.pi*u2)
    q3 = np.sqrt(u1)   * np.sin(2*np.pi*u3)
    q4 = np.sqrt(u1)   * np.cos(2*np.pi*u3)
    # quaternion (q1,q2,q3,q4) -> R
    x,y,z,w = q1,q2,q3,q4
    Rm = np.array([
        [1-2*(y*y+z*z), 2*(x*y - z*w), 2*(x*z + y*w)],
        [2*(x*y + z*w), 1-2*(x*x+z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w), 2*(y*z + x*w), 1-2*(x*x+y*y)],
    ], dtype=np.float32)
    return Rm

def _center(coords):
    c = np.asarray(coords, dtype=np.float32)
    return c - c.mean(axis=0, keepdims=True)

def make_5_predictions(coords_base, seq, seed=0):
    """
    coords_base: (N,3) float
    seq: string
    Returns list of 5 arrays (N,3) float32
    """
    rng = np.random.default_rng(int(seed))
    base = _center(coords_base).astype(np.float32)

    preds = []

    # helper: apply rigid + noise then refine
    def build(rot_scale=1.0, trans_scale=0.0, noise=0.0):
        x = base.copy()
        if rot_scale > 0:
            Rm = _rand_rotation(rng)
            # interpolate towards identity if rot_scale < 1
            if rot_scale < 1.0:
                I = np.eye(3, dtype=np.float32)
                Rm = (1.0-rot_scale)*I + rot_scale*Rm
            x = x @ Rm.T
        if trans_scale > 0:
            t = rng.normal(0, trans_scale, size=(1,3)).astype(np.float32)
            x = x + t
        if noise > 0:
            x = x + rng.normal(0, noise, size=x.shape).astype(np.float32)
        # light refine (gentle)
        x = light_refine_coords(
            x, seq,
            n_iter=3,
            smooth_w=0.10,
            dist_min=5.4, dist_max=6.6, dist_target=6.0,
            dist_strength=0.30,
            min_nonseq=3.8,
            clash_strength=0.22,
            clash_window=40
        )
        return x

    # 1) base refined
    preds.append(build(rot_scale=0.0, noise=0.0))

    # 2) gentle noise
    preds.append(build(rot_scale=0.0, noise=0.10))

    # 3) slightly stronger noise
    preds.append(build(rot_scale=0.0, noise=0.18))

    # 4) random rotation + gentle noise
    preds.append(build(rot_scale=1.0, noise=0.10))

    # 5) random rotation + moderate noise
    preds.append(build(rot_scale=1.0, noise=0.18))

    return preds

# ---- smoke test using current mapped/refined coords from previous stages ----
print("=== STAGE 9: Best-of-5 Strategy ===")

# prefer already refined coords_ref from STAGE 8, else res["coords"], else remap from df_rank
coords0 = None
seq0 = None
tpl0 = None

if "coords_ref" in globals():
    coords0 = coords_ref
if coords0 is None and "res" in globals() and isinstance(res, dict) and "coords" in res:
    coords0 = res["coords"]
    tpl0 = res.get("template_id", None)
if "_seq" in globals():
    seq0 = _seq

if coords0 is None:
    if "mapped_coords_from_best" in globals() and "df_rank" in globals():
        COMP_ROOT = "/kaggle/input/stanford-rna-3d-folding-2"
        _t = pd.read_csv(f"{COMP_ROOT}/test_sequences.csv", usecols=["target_id","sequence"], dtype=str, low_memory=False).head(1)
        _tid = _t.loc[0, "target_id"]
        seq0 = _t.loc[0, "sequence"]
        rr = mapped_coords_from_best(df_rank, seq0, use_cache=True)
        tpl0 = rr["template_id"]
        coords0 = light_refine_coords(rr["coords"], seq0)
    else:
        raise RuntimeError("No base coords found. Jalankan STAGE 7 dan 8 dulu.")

preds5 = make_5_predictions(coords0, seq0, seed=42)

# quick diversity check: RMSD vs pred1 (after centering)
def _rmsd(a,b):
    a = _center(a); b=_center(b)
    return float(np.sqrt(((a-b)**2).mean()))

rmsds = [_rmsd(preds5[0], preds5[i]) for i in range(5)]
print(f"N={preds5[0].shape[0]} | RMSD vs pred1: {[round(x,3) for x in rmsds]}")
print("First residue coords (5 preds):")
for i in range(5):
    print(f"  pred{i+1}: {preds5[i][0]}")


=== STAGE 9: Best-of-5 Strategy ===
N=30 | RMSD vs pred1: [0.0, 0.054, 0.116, 15.888, 11.111]
First residue coords (5 preds):
  pred1: [ -1.9953877 -15.065442   21.181362 ]
  pred2: [ -1.9679787 -15.164508   21.161325 ]
  pred3: [ -1.8763458 -15.073864   21.000858 ]
  pred4: [ 21.315096   -0.7087547 -15.005429 ]
  pred5: [ -1.2297273 -25.048454   -7.644714 ]


# Validation Driven Tuning 

In [10]:
# ============================================================
# STAGE 10 — Validation Driven Tuning (ONE CELL)
# - Build GT references from validation_labels (multi-conformation)
# - Evaluate avg best-of-5 TM-score on validation targets
# - Tune Best-of-5 params (noise/refine strength) with a small grid
# - Quiet: no warning spam
#
# Requires (from your previous stages):
#   - df_tpl (template_library) [or /kaggle/working/template_library.parquet]
#   - get_c1_for_template (STAGE 6)
#   - map_coords_via_alignment (STAGE 7)
#   - light_refine_coords (STAGE 8)
# Optional:
#   - df_rank (STAGE 5 output) for current target; if missing, this stage will not rebuild ranking.
#
# Output:
#   - best_cfg (dict)
#   - tuning_table (DataFrame)
#   - saves: /kaggle/working/tuning_best_cfg.json
# ============================================================

import warnings, json, math
from pathlib import Path
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=pd.errors.DtypeWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
try:
    from Bio import BiopythonDeprecationWarning, BiopythonWarning
    warnings.filterwarnings("ignore", category=BiopythonDeprecationWarning)
    warnings.filterwarnings("ignore", category=BiopythonWarning)
except Exception:
    pass

COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
VAL_SEQ_PATH = COMP_ROOT / "validation_sequences.csv"
VAL_LBL_PATH = COMP_ROOT / "validation_labels.csv"

assert VAL_SEQ_PATH.exists(), f"Missing: {VAL_SEQ_PATH}"
assert VAL_LBL_PATH.exists(), f"Missing: {VAL_LBL_PATH}"

need = ["get_c1_for_template", "map_coords_via_alignment", "light_refine_coords"]
for n in need:
    if n not in globals():
        raise RuntimeError(f"Missing '{n}'. Jalankan STAGE terkait dulu: {n}")

# ---- TM-score (Kabsch + TM formula) ----
def _kabsch(P, Q):
    # P,Q shape (N,3), returns R (3,3) s.t. P@R ~ Q
    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)
    C = Pc.T @ Qc
    V, S, Wt = np.linalg.svd(C)
    d = np.sign(np.linalg.det(V @ Wt))
    D = np.diag([1.0, 1.0, d])
    R = V @ D @ Wt
    return R.astype(np.float32)

def tm_score(pred, ref):
    # pred/ref (N,3) same N. Align pred to ref.
    P = np.asarray(pred, dtype=np.float32)
    Q = np.asarray(ref, dtype=np.float32)
    n = P.shape[0]
    if n == 0:
        return 0.0
    # d0 (classic TM-score heuristic; safe clamp for small n)
    L = float(n)
    d0 = 1.24 * max(L - 15.0, 1.0) ** (1.0/3.0) - 1.8
    d0 = float(max(d0, 0.5))
    R = _kabsch(P, Q)
    Pc = P - P.mean(axis=0, keepdims=True)
    Qc = Q - Q.mean(axis=0, keepdims=True)
    A = Pc @ R
    d = np.sqrt(((A - Qc) ** 2).sum(axis=1))
    score = np.mean(1.0 / (1.0 + (d / d0) ** 2))
    return float(score)

def best_of_5_tm(preds5, refs):
    # preds5: list of 5 (N,3); refs: list of (N,3)
    best = 0.0
    for pr in preds5:
        for rf in refs:
            best = max(best, tm_score(pr, rf))
    return best

# ---- Load validation sequences ----
df_val = pd.read_csv(VAL_SEQ_PATH, dtype=str, low_memory=False)[["target_id","sequence","temporal_cutoff"]]
df_val["temporal_cutoff"] = pd.to_datetime(df_val["temporal_cutoff"], errors="coerce")

# ---- Build GT reference coords from validation_labels (multi-conformation) ----
val_lbl = pd.read_csv(VAL_LBL_PATH, low_memory=False)
# parse target_id prefix from ID = "<target>_<resid>"
val_lbl["target_id"] = val_lbl["ID"].astype(str).str.rsplit("_", n=1).str[0]
val_lbl = val_lbl.sort_values(["target_id","resid"])

coord_cols = [c for c in val_lbl.columns if c[0] in ("x","y","z") and "_" in c]
# suffix indices available
idxs = sorted({int(c.split("_")[1]) for c in coord_cols if c.split("_")[1].isdigit()})

def build_refs_for_target(tid):
    g = val_lbl[val_lbl["target_id"] == tid]
    if len(g) == 0:
        return []
    refs = []
    for k in idxs:
        cx, cy, cz = f"x_{k}", f"y_{k}", f"z_{k}"
        if cx not in g.columns or cy not in g.columns or cz not in g.columns:
            continue
        arr = g[[cx,cy,cz]].to_numpy(dtype=np.float32, copy=True)
        # skip empty/NaN refs
        if not np.isfinite(arr).any():
            continue
        if np.isnan(arr).any():
            arr = np.nan_to_num(arr, nan=0.0)
        refs.append(arr)
    return refs

gt_refs = {tid: build_refs_for_target(tid) for tid in df_val["target_id"].tolist()}

# ---- Base coords precompute (minimal; assumes you already have best template+alignment strings somewhere) ----
# Strategy:
# - If you already computed per-target ranking elsewhere, cache them into base_cache yourself.
# - Here we support a simple fallback:
#   - If a global df_rank exists and matches the current target_id, use it.
#   - Otherwise we skip that target (so tuning doesn't crash).
base_cache = {}  # tid -> {"seq":..., "template_id":..., "coords_base": (N,3)}

def _same_target_df_rank(df_rank, tid):
    # Heuristic: if top template starts with same PDB id as tid (often true), accept.
    try:
        top = str(df_rank.iloc[0]["template_id"])
        return True if top else False
    except Exception:
        return False

for _, r in df_val.iterrows():
    tid = r["target_id"]
    seq = r["sequence"]
    refs = gt_refs.get(tid, [])
    if not refs:
        continue

    # try use existing df_rank (user usually runs ranking per target before this stage)
    if "df_rank" not in globals() or not isinstance(df_rank, pd.DataFrame) or len(df_rank) == 0:
        continue
    if not _same_target_df_rank(df_rank, tid):
        # If your df_rank is for a different target, skip (no crash).
        continue

    rr0 = df_rank.iloc[0]
    tpl_id = rr0["template_id"]
    aligned_q = rr0["aligned_q"]
    aligned_t = rr0["aligned_t"]

    tpl = get_c1_for_template(tpl_id, use_cache=True, debug=False)
    if tpl is None:
        continue

    mapped = map_coords_via_alignment(seq, tpl["coords"], aligned_q, aligned_t)
    coords_base = light_refine_coords(mapped, seq)

    base_cache[tid] = {"seq": seq, "template_id": tpl_id, "coords_base": coords_base}

# ---- Parametric best-of-5 generator (tunable) ----
def _rand_rotation(rng):
    u1, u2, u3 = rng.random(3)
    q1 = np.sqrt(1-u1) * np.sin(2*np.pi*u2)
    q2 = np.sqrt(1-u1) * np.cos(2*np.pi*u2)
    q3 = np.sqrt(u1)   * np.sin(2*np.pi*u3)
    q4 = np.sqrt(u1)   * np.cos(2*np.pi*u3)
    x,y,z,w = q1,q2,q3,q4
    Rm = np.array([
        [1-2*(y*y+z*z), 2*(x*y - z*w), 2*(x*z + y*w)],
        [2*(x*y + z*w), 1-2*(x*x+z*z), 2*(y*z - x*w)],
        [2*(x*z - y*w), 2*(y*z + x*w), 1-2*(x*x+y*y)],
    ], dtype=np.float32)
    return Rm

def _center(x):
    x = np.asarray(x, dtype=np.float32)
    return x - x.mean(axis=0, keepdims=True)

def make_5_predictions_cfg(coords_base, seq, seed, cfg):
    rng = np.random.default_rng(int(seed))
    base = _center(coords_base).astype(np.float32)

    def build(rot, noise):
        x = base.copy()
        if rot:
            x = x @ _rand_rotation(rng).T
        if noise > 0:
            x = x + rng.normal(0, noise, size=x.shape).astype(np.float32)
        x = light_refine_coords(
            x, seq,
            n_iter=int(cfg["n_iter"]),
            smooth_w=float(cfg["smooth_w"]),
            dist_min=5.4, dist_max=6.6, dist_target=6.0,
            dist_strength=float(cfg["dist_strength"]),
            min_nonseq=3.8,
            clash_strength=float(cfg["clash_strength"]),
            clash_window=40,
        )
        return x.astype(np.float32)

    n1 = float(cfg["noise_small"])
    n2 = float(cfg["noise_big"])
    return [
        build(rot=False, noise=0.0),
        build(rot=False, noise=n1),
        build(rot=False, noise=n2),
        build(rot=True,  noise=n1),
        build(rot=True,  noise=n2),
    ]

# ---- tuning grid (small & safe) ----
grid = []
for smooth_w in [0.08, 0.10, 0.12]:
    for dist_strength in [0.22, 0.30, 0.38]:
        grid.append({
            "n_iter": 3,
            "smooth_w": smooth_w,
            "dist_strength": dist_strength,
            "clash_strength": 0.22,
            "noise_small": 0.10,
            "noise_big": 0.18,
        })
# a couple noise variants
grid += [
    {**grid[0], "noise_small": 0.08, "noise_big": 0.16},
    {**grid[0], "noise_small": 0.12, "noise_big": 0.22},
]

# ---- evaluate ----
rows = []
targets = sorted(list(base_cache.keys()))
if len(targets) == 0:
    raise RuntimeError(
        "Base cache kosong. Artinya df_rank kamu saat ini tidak cocok untuk target validation.\n"
        "Solusi: jalankan STAGE 5 (ranking) per target validation dulu, atau simpan df_rank per target lalu load di sini."
    )

for gi, cfg in enumerate(grid):
    scores = []
    for tid in targets:
        seq = base_cache[tid]["seq"]
        refs = gt_refs[tid]
        preds5 = make_5_predictions_cfg(base_cache[tid]["coords_base"], seq, seed=42, cfg=cfg)
        s = best_of_5_tm(preds5, refs)
        scores.append(s)
    rows.append({
        "grid_i": gi,
        "avg_bestof5_tm": float(np.mean(scores)),
        "min_tm": float(np.min(scores)),
        "cfg": cfg
    })

tuning_table = pd.DataFrame(rows).sort_values("avg_bestof5_tm", ascending=False).reset_index(drop=True)
best_cfg = tuning_table.loc[0, "cfg"]
best_score = tuning_table.loc[0, "avg_bestof5_tm"]

Path("/kaggle/working/tuning_best_cfg.json").write_text(json.dumps({"avg_bestof5_tm": best_score, "cfg": best_cfg}, indent=2))
print("=== STAGE 10: Validation Driven Tuning ===")
print(f"Targets evaluated: {len(targets)}")
print(f"Best avg best-of-5 TM: {best_score:.4f}")
print("Best cfg:", best_cfg)
display(tuning_table.head(8))


=== STAGE 10: Validation Driven Tuning ===
Targets evaluated: 28
Best avg best-of-5 TM: 0.1267
Best cfg: {'n_iter': 3, 'smooth_w': 0.08, 'dist_strength': 0.38, 'clash_strength': 0.22, 'noise_small': 0.1, 'noise_big': 0.18}


,grid_i,avg_bestof5_tm,min_tm,cfg
0,2,0.126722,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.08, 'dist_strength..."
1,10,0.126628,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.08, 'dist_strength..."
2,1,0.126586,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.08, 'dist_strength..."
3,0,0.126567,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.08, 'dist_strength..."
4,9,0.126536,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.08, 'dist_strength..."
5,5,0.126321,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.1, 'dist_strength'..."
6,4,0.126291,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.1, 'dist_strength'..."
7,3,0.126235,1.606915e-22,"{'n_iter': 3, 'smooth_w': 0.1, 'dist_strength'..."


# Robust Submission Build & Sanity Checks

In [11]:
# ============================================================
# STAGE 11 — Robust Submission Build & Sanity Checks (ONE CELL) [REVISI FULL v3]
# Disesuaikan dengan alur kode kamu:
# - Prioritas sumber prediksi:
#   (1) submission_df (DataFrame)
#   (2) all_predictions (list of dict)
#   (3) preds_by_target (dict: target_id -> list 5 array (L,3))
#   (4) file /kaggle/working/submission.csv atau ./submission.csv (fallback)
#   (5) jika belum ada semuanya, dan ada predict_rna_structures + (test_seqs, train_seqs, train_coords_dict)
#       => otomatis generate prediksi (tanpa plot) lalu build submission
#   (6) terakhir: fallback submission nol (tidak error)
# - Align persis ke sample_submission.csv (order + jumlah baris)
# - Fill missing dengan 0, clip ke [-999.999, 9999.999], finite-safe
# Output final: /kaggle/working/submission.csv
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from pathlib import Path

COMP_ROOT = Path("/kaggle/input/stanford-rna-3d-folding-2")
SAMPLE_PATH = COMP_ROOT / "sample_submission.csv"
TEST_SEQ_PATH = COMP_ROOT / "test_sequences.csv"
OUT_PATH = Path("/kaggle/working/submission.csv")

assert SAMPLE_PATH.exists(), f"Missing: {SAMPLE_PATH}"
assert TEST_SEQ_PATH.exists(), f"Missing: {TEST_SEQ_PATH}"

COORD_COLS = [f"{a}_{i}" for i in range(1, 6) for a in ("x","y","z")]
REQ_COLS = ["ID","resname","resid"] + COORD_COLS

def _clip_arr(x: np.ndarray) -> np.ndarray:
    return np.clip(x, -999.999, 9999.999)

def _build_from_preds_by_target(preds_by_target: dict, seq_map: dict):
    rows = []
    for tid, preds5 in preds_by_target.items():
        tid = str(tid)
        seq = str(seq_map.get(tid, ""))
        L = len(seq)

        # normalize preds5 to list length 5
        if isinstance(preds5, np.ndarray) and preds5.ndim == 3 and preds5.shape[0] == 5:
            preds5 = [preds5[i] for i in range(5)]
        if not isinstance(preds5, (list, tuple)) or len(preds5) < 5:
            continue

        arrs = []
        for k in range(5):
            a = np.asarray(preds5[k], dtype=np.float32)
            if a.ndim != 2 or a.shape[1] != 3:
                a = None
            arrs.append(a)

        for r in range(1, L + 1):
            row = {"ID": f"{tid}_{r}", "resname": seq[r-1], "resid": r}
            for k in range(5):
                if arrs[k] is None or (r-1) >= arrs[k].shape[0]:
                    x = y = z = 0.0
                else:
                    x, y, z = map(float, arrs[k][r-1])
                row[f"x_{k+1}"] = x
                row[f"y_{k+1}"] = y
                row[f"z_{k+1}"] = z
            rows.append(row)
    return pd.DataFrame(rows) if rows else None

def _ensure_cols(pred_df: pd.DataFrame, seq_map: dict):
    pred_df = pred_df.copy()
    pred_df["ID"] = pred_df["ID"].astype("string")

    # if resname/resid missing -> reconstruct from ID + test sequence
    if "resid" not in pred_df.columns or "resname" not in pred_df.columns:
        def _resname_from_id(s):
            s = str(s)
            if "_" not in s:
                return "A"
            t, r = s.rsplit("_", 1)
            try:
                r = int(r)
            except Exception:
                return "A"
            seq = str(seq_map.get(t, ""))
            if seq and 1 <= r <= len(seq):
                return seq[r-1]
            return "A"

        def _resid_from_id(s):
            s = str(s)
            if "_" not in s:
                return 0
            try:
                return int(s.rsplit("_", 1)[1])
            except Exception:
                return 0

        if "resid" not in pred_df.columns:
            pred_df["resid"] = pred_df["ID"].astype(str).map(_resid_from_id).astype(int)
        if "resname" not in pred_df.columns:
            pred_df["resname"] = pred_df["ID"].astype(str).map(_resname_from_id)

    for c in COORD_COLS:
        if c not in pred_df.columns:
            pred_df[c] = 0.0

    # drop duplicate IDs (keep first)
    if pred_df["ID"].duplicated().any():
        pred_df = pred_df.drop_duplicates("ID", keep="first")

    return pred_df

# ---- load sample + test sequences ----
sample = pd.read_csv(SAMPLE_PATH, dtype={"ID":"string"}, low_memory=False)[["ID","resname","resid"]].copy()
df_test = pd.read_csv(TEST_SEQ_PATH, dtype=str, low_memory=False)[["target_id","sequence"]]
seq_map = dict(zip(df_test["target_id"].astype(str), df_test["sequence"].astype(str)))

pred_df = None
pred_source = None

# (1) submission_df
if "submission_df" in globals() and isinstance(globals()["submission_df"], pd.DataFrame) and len(submission_df) > 0:
    pred_df = submission_df.copy()
    pred_source = "submission_df"

# (2) all_predictions
elif "all_predictions" in globals() and isinstance(globals()["all_predictions"], list) and len(all_predictions) > 0:
    pred_df = pd.DataFrame(all_predictions)
    pred_source = "all_predictions"

# (3) preds_by_target
elif "preds_by_target" in globals() and isinstance(globals()["preds_by_target"], dict) and len(preds_by_target) > 0:
    pred_df = _build_from_preds_by_target(preds_by_target, seq_map)
    pred_source = "preds_by_target"

# (4) existing submission.csv file
if pred_df is None or len(pred_df) == 0:
    cand_paths = [OUT_PATH, Path("/kaggle/working/submission.csv"), Path("submission.csv")]
    found = next((p for p in cand_paths if p.exists()), None)
    if found is not None:
        pred_df = pd.read_csv(found, dtype={"ID":"string"}, low_memory=False)
        pred_source = f"file:{str(found)}"

# (5) AUTO-GENERATE using your earlier function if available
if (pred_df is None or len(pred_df) == 0) and ("predict_rna_structures" in globals()):
    # Need these to exist (sesuai kode kamu)
    req = ["test_seqs", "train_seqs", "train_coords_dict"]
    if all((r in globals()) for r in req):
        rows = []
        for _, row in test_seqs.iterrows():
            tid = str(row["target_id"])
            seq = str(row["sequence"])
            tcut = row["temporal_cutoff"] if "temporal_cutoff" in row else None

            preds5 = predict_rna_structures(seq, tid, train_seqs, train_coords_dict, n_predictions=5, temporal_cutoff=tcut)

            for j in range(len(seq)):
                rec = {"ID": f"{tid}_{j+1}", "resname": seq[j], "resid": j+1}
                for i in range(5):
                    rec[f"x_{i+1}"] = float(preds5[i][j][0])
                    rec[f"y_{i+1}"] = float(preds5[i][j][1])
                    rec[f"z_{i+1}"] = float(preds5[i][j][2])
                rows.append(rec)

        pred_df = pd.DataFrame(rows)
        pred_source = "auto:predict_rna_structures"

# (6) FINAL FALLBACK: zeros (no error)
if pred_df is None or len(pred_df) == 0:
    pred_df = sample.copy()
    for c in COORD_COLS:
        pred_df[c] = 0.0
    pred_source = "fallback:zeros"

# ---- normalize columns ----
pred_df = _ensure_cols(pred_df, seq_map)

# ---- align to sample order ----
sub = sample.merge(pred_df[["ID"] + COORD_COLS], on="ID", how="left")

# fill NaNs with 0 + float32
for c in COORD_COLS:
    sub[c] = sub[c].fillna(0.0).astype(np.float32)

# clip + finite sanitize
X = sub[COORD_COLS].to_numpy(dtype=np.float32, copy=True)
X = _clip_arr(X)
X = np.nan_to_num(X, nan=0.0, posinf=9999.999, neginf=-999.999).astype(np.float32)
sub[COORD_COLS] = X

sub = sub[REQ_COLS]

# ---- sanity checks ----
id_ok = (sub["ID"].astype(str).values == sample["ID"].astype(str).values).all()
finite_ok = bool(np.isfinite(sub[COORD_COLS].to_numpy()).all())
minv = float(sub[COORD_COLS].to_numpy().min())
maxv = float(sub[COORD_COLS].to_numpy().max())
zero_frac = float((sub[COORD_COLS].to_numpy() == 0).mean())
dup_ids = int(sub["ID"].duplicated().sum())

print("=== STAGE 11: Robust Submission Build & Sanity Checks (v3) ===")
print(f"pred_source: {pred_source}")
print(f"rows: {len(sub)} | sample_rows: {len(sample)} | id_order_match={id_ok} | finite_ok={finite_ok} | dup_ids={dup_ids}")
print(f"coord_range: [{minv:.3f}, {maxv:.3f}] | zero_frac={zero_frac:.4f}")
print("columns_ok:", sub.columns.tolist() == REQ_COLS)
print(sub.head(3))

# ---- save ----
sub.to_csv(OUT_PATH, index=False)
Path("submission.csv").write_text(OUT_PATH.read_text())
print(f"Saved final submission: {OUT_PATH}")


=== STAGE 11: Robust Submission Build & Sanity Checks (v3) ===
pred_source: fallback:zeros
rows: 9762 | sample_rows: 9762 | id_order_match=True | finite_ok=True | dup_ids=0
coord_range: [0.000, 0.000] | zero_frac=1.0000
columns_ok: True
       ID resname  resid  x_1  y_1  z_1  x_2  y_2  z_2  x_3  y_3  z_3  x_4  \
0  8ZNQ_1       A      1  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
1  8ZNQ_2       C      2  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   
2  8ZNQ_3       C      3  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0   

   y_4  z_4  x_5  y_5  z_5  
0  0.0  0.0  0.0  0.0  0.0  
1  0.0  0.0  0.0  0.0  0.0  
2  0.0  0.0  0.0  0.0  0.0  
Saved final submission: /kaggle/working/submission.csv
